# SERFF fair value: ZQ vs SR3 residual-ledger backtest

Three-layer SOFR−FF model (policy path / reserve-demand level / turn hurdle) emitting a dated, source-tagged residual ledger per decision date, backtested on a DV01-neutral 5 SR3 : 3 ZQ strip structure.

Spec: `docs/superpowers/specs/2026-07-02-serff-fair-value-model.md` — pairs with `BT/serff/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import datetime
import pandas as pd

from BT.serff.config import SerffBacktestConfig, SerffDataConfig, SerffModelConfig
from BT.serff.data import build_panel
from BT.serff.layers import fit_layers

## 1. Panel + prototype-parity fits (contemporaneous alignment)

In [ ]:
data_cfg = SerffDataConfig(start=datetime.date(2018, 4, 2), alignment="contemporaneous")
model_cfg = SerffModelConfig()
panel = build_panel(data_cfg, model_cfg)
fit = fit_layers(panel, model_cfg)
print(f"panel {panel.shape}, regimes:")
display(fit.level.per_regime.round(3))
print("logit z:", dict(zip(fit.turn.logit.params.index, fit.turn.logit.tvalues.round(2))))
ln_liq = float(panel['ln_liq'].iloc[-1])
print("month-end:", fit.turn.predict(ln_liq, False))
print("quarter-end:", fit.turn.predict(ln_liq, True))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for rg, g in panel[~panel['turn_window']].groupby('regime'):
    axes[0].scatter(g['liq_gdp'], g['spread'], s=4, alpha=0.4, label=rg)
axes[0].set_xlabel('liq/GDP %'); axes[0].set_ylabel('SOFR-FF bp'); axes[0].legend(fontsize=7)
axes[0].set_title('reserve demand curve by regime')
me = panel[panel['is_me']]
axes[1].scatter(me['liq_gdp'], me['spike'], s=14, c=me['is_qe'].map({True: 'tab:red', False: 'tab:blue'}))
axes[1].axhline(model_cfg.spike_threshold_bp, ls='--', c='gray')
axes[1].set_title('month-end spikes (red = quarter-end)'); axes[1].set_xlabel('liq/GDP %')
plt.tight_layout()

## 2. Current residual ledger (dated, source-tagged)

In [ ]:
from BT.serff.futures_data import backfill_settles, settle_panel
from BT.serff.ledger import build_ledger
from BT.serff.mechanics import fomc_decision_dates, covering_zq_months, contract_window, zq_monthly_symbols
from BT.serff.data import load_fixings

today = datetime.date.today()
hist = backfill_settles(datetime.date(2018, 4, 2), today, show_progress=False)
settles = settle_panel(hist)
sofr, effr = load_fixings('USD-SOFR-1D'), load_fixings('USD-FEDFUNDS')

ts = settles.index.max()
t = ts.date()
sr3_sym = 'SR3U26'  # or use BT.serff.backtest._active_sr3
need = [sr3_sym] + [s for s, _ in covering_zq_months(contract_window(sr3_sym))] + zq_monthly_symbols(t, contract_window(sr3_sym).end)
prices = {s: float(settles.at[ts, s]) for s in dict.fromkeys(need) if s in settles.columns and pd.notna(settles.at[ts, s])}
ledger = build_ledger(t, sr3_symbol=sr3_sym, prices=prices,
                      sofr_fixings=sofr[sofr.index < ts], effr_fixings=effr[effr.index < ts],
                      fit=fit, covariates={'ln_liq': ln_liq, 'tga_gdp': float(panel['tga_gdp'].iloc[-1])},
                      meeting_decisions=fomc_decision_dates(datetime.date(2015,1,1), datetime.date(2030,1,1)), cfg=model_cfg)
print('structure residual (basis+turn):', round(ledger.structure_residual_bp, 2), 'bp')
print('by source:', {k: round(v, 2) for k, v in ledger.residual_by_source.items()})
display(ledger.policy_exposure.round(4))
ledger.sr3.daily[ledger.sr3.daily['status'] == 'forward'].head(12)

## 3. Walk-forward backtest with source attribution

In [ ]:
from BT.serff.backtest import run_serff_backtest, plot_pnl

cfg = SerffBacktestConfig()
result = run_serff_backtest(cfg, settles=settles, sofr_fixings=sofr, effr_fixings=effr)
print(result.summary)
display(result.per_regime.round(0))
plot_pnl(result);

In [ ]:
# turn-model calibration out of sample + settle reconciliation gate
te = result.turn_events
display(te.groupby('is_qe')[['p_hit_model', 'hit_realized']].mean().round(3))
recon = result.reconciliation
print('reconciliation within tolerance:', int(recon['ok'].sum()), '/', len(recon))
recon[~recon['ok']].head(10)